In [1]:
import cv2
import numpy as np
import os

input_folder = "input"
output_folder = "output"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_folder, filename)
    image = cv2.imread(image_path)

    if image is None:
        continue

    image = cv2.resize(image, (800, 800))

    # =========================
    # 1. EDGE-BASED PREPROCESSING
    # =========================
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Improve edge quality
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Auto-Canny (better than fixed thresholds)
    median_val = np.median(blur)
    lower = int(max(0, 0.66 * median_val))
    upper = int(min(255, 1.33 * median_val))

    edges = cv2.Canny(blur, lower, upper)

    # =========================
    # 2. CLEAN OUTLINES
    # =========================
    kernel = np.ones((5, 5), np.uint8)

    # Close gaps in outlines
    edges = cv2.dilate(edges, kernel, iterations=2)
    edges = cv2.erode(edges, kernel, iterations=1)

    # Fill small breaks
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    # =========================
    # 3. FIND CONTOURS
    # =========================
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    intact = 0
    broken = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        # Filter noise and background
        if area < 300 or area > 200000:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        if w > 750 or h > 750:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        # =========================
        # 4. OUTLINE FEATURES
        # =========================

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            continue

        solidity = float(area) / hull_area

        # =========================
        # 5. CLASSIFICATION
        # =========================

        if circularity > 0.75 and solidity > 0.50:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1
        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

        # Draw outline + bounding box
        cv2.drawContours(image, [cnt], -1, color, 2)
        cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)

        cv2.putText(image, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # =========================
    # 6. SUMMARY
    # =========================
    cv2.putText(image, f"Intact: {intact}  Broken: {broken}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    # =========================
    # 7. SAVE OUTPUT
    # =========================
    output_path = os.path.join(output_folder, f"processed_{filename}")
    cv2.imwrite(output_path, image)

print("Outline-based processing complete. Check output_images folder.")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'input'